# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### 📋 Operationalizing ML Probabilities
Instead of presenting raw statistical decimal values (like `0.84`) that content writers might find confusing, our playbook translates machine learning probabilities into dynamic, human-interpretable priority bands and structured **Reason Codes**:

1. **`CRITICAL_NEGLECT`**: High-probability decay pages that have not been modified or touched in a long time (e.g., >250 days). These require immediate factual reviews and structured rewrites.
2. **`HIGH_EXPOSURE_RANK_DECAY`**: Core, high-impression pages that are actively slipping down search ranking positions. These require urgent optimization to prevent major organic traffic losses.
3. **`STANDARD_DECAY_RISK`**: Pages with moderate to high decay probability showing early signs of traffic decline. These are slotted into standard weekly editorial cycles.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# 1. Clone repository in Google Colab to access data directory
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/rafayyk7/flyrank-ml.git"
    REPO_DIR = "flyrank-ml"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository into Google Colab...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# 2. Establish path and load the data
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# Impute missing values
features_list = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
for col in features_list:
    df[col] = df[col].fillna(df[col].median())

df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 3. Train Grouped model
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, test_idx = next(gss.split(df, df['is_declining_label'], groups=df['client_id']))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(train_df[features_list], train_df['is_declining_label'])

# Generate predictions on the holdout set
test_df['ml_decay_prob'] = rf_model.predict_proba(test_df[features_list])[:, 1]

# 4. Generate human-readable reason codes based on feature thresholds
reason_codes = []
for idx, row in test_df.iterrows():
    if row['days_since_last_update'] > 250:
        reason_codes.append("CRITICAL_NEGLECT")
    elif row['avg_position'] > 15 and row['impressions_90d'] > test_df['impressions_90d'].median():
        reason_codes.append("HIGH_EXPOSURE_RANK_DECAY")
    else:
        reason_codes.append("STANDARD_DECAY_RISK")

test_df['reason_code'] = reason_codes

# Sort descending by prediction confidence to create the ranked queue
ranked_queue = test_df.sort_values(by='ml_decay_prob', ascending=False).copy()
ranked_queue['priority_rank'] = range(1, len(ranked_queue) + 1)

print("--- TOP 10 PRIORITIZED ACTIONS ---")
print(ranked_queue[['priority_rank', 'content_id', 'ml_decay_prob', 'reason_code', 'days_since_last_update', 'avg_position']].head(10).to_markdown(index=False))

Cloning repository into Google Colab...
--- TOP 10 PRIORITIZED ACTIONS ---
|   priority_rank | content_id           |   ml_decay_prob | reason_code              |   days_since_last_update |   avg_position |
|----------------:|:---------------------|----------------:|:-------------------------|-------------------------:|---------------:|
|               1 | content_2a228ce7aa1b |        0.821582 | STANDARD_DECAY_RISK      |                       92 |            7.5 |
|               2 | content_f55fd2d8ed04 |        0.816799 | STANDARD_DECAY_RISK      |                      104 |            1.3 |
|               3 | content_884c401ce126 |        0.816049 | STANDARD_DECAY_RISK      |                       92 |           23.1 |
|               4 | content_3395aba722a2 |        0.813205 | STANDARD_DECAY_RISK      |                      102 |           11.1 |
|               5 | content_32b4e5a2f630 |        0.812292 | STANDARD_DECAY_RISK      |                      102 |            6.5 |
|

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

*   **Intended Use Case:** This prioritized ranking queue is built for content editors, SEO product managers, and copywriters. It should be used during weekly or monthly content planning sessions to optimize the allocation of writing budgets by targeting decaying pages.
*   **Where it stops being valid (System Limits):**
    1. **Highly Seasonal Events:** The model relies on raw historical performance drops. It cannot distinguish seasonal fluctuations (e.g., tax preparation tips dipping in summer) from structural page decay.
    2. **Cold Start Issues:** For newly onboarded clients or freshly launched URLs with under 90 days of Google Search Console footprint, the feature signals are highly volatile and unreliable.
    3. **Structural Migrations:** If a client changes their domain URL structures or site framework, historical tracking metrics lose their direct semantic links, rendering old predictions invalid.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatically flag and filter out cold-start pages (e.g., extremely new pages with low age)
cold_start_threshold = 90
cold_start_count = (ranked_queue['content_age_days'] < cold_start_threshold).sum()

print("--- System Limits Diagnostics ---")
print(f"Total pages in operational queue: {len(ranked_queue):,}")
print(f"Pages flagged as cold-start warning (< {cold_start_threshold} days old): {cold_start_count:,}")
print(f"Action: Safe exclusion filter applied. Proceeding with high-confidence assets.")

--- System Limits Diagnostics ---
Total pages in operational queue: 6,163
Pages flagged as cold-start warning (< 90 days old): 0
Action: Safe exclusion filter applied. Proceeding with high-confidence assets.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 🛡️ The Operational Guardrails
Our machine learning model is an advisor, not an automation tool. Editors must perform manual inspections on high-probability pages before initiating a rewrite.

*   **Human Review Checklist (Before Writing):**
    1. **Cannibalization Check:** Search the main target keywords manually. Is another page on the client's site now ranking higher for this term? If yes, a rewrite will cause rank cannibalization; redirect or consolidate instead.
    2. **SERP Intent Shift:** Check if Google has redesigned the results page for the target query to favor videos, shopping links, or local maps. If search intent has shifted, text-based updates will not recover rank.
    3. **Existing Factual Accuracy:** Ensure the page is actually outdated before rewriting.

*   **The No-Go List (Never Automate):**
    *   *Do not* plug high-decay URLs into automated generative AI tools to write and publish content without strict human-in-the-loop editorial review.
    *   *Do not* trigger automatic 301 redirects or page deletions based on high-probability decay scores alone.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Isolate high-impact pages that absolutely require manual cannibalization review (High impressions, poor rank)
review_candidates = ranked_queue[
    (ranked_queue['impressions_90d'] > ranked_queue['impressions_90d'].quantile(0.90)) &
    (ranked_queue['avg_position'] > 20)
]

print(f"--- HUMAN REVIEW EXCLUSION AUDIT ---")
print(f"Found {len(review_candidates):,} high-impact pages requiring mandatory manual SERP and cannibalization audits.")
print("Sample list of top candidates for human-in-the-loop review:")
print(review_candidates[['content_id', 'impressions_90d', 'avg_position', 'reason_code']].head(5).to_markdown(index=False))

--- HUMAN REVIEW EXCLUSION AUDIT ---
Found 76 high-impact pages requiring mandatory manual SERP and cannibalization audits.
Sample list of top candidates for human-in-the-loop review:
| content_id           |   impressions_90d |   avg_position | reason_code              |
|:---------------------|------------------:|---------------:|:-------------------------|
| content_422ea7d8363a |             11795 |           29.9 | HIGH_EXPOSURE_RANK_DECAY |
| content_e09b5602ba42 |              9681 |           74.3 | HIGH_EXPOSURE_RANK_DECAY |
| content_b45f91ebc732 |             14940 |           59.4 | HIGH_EXPOSURE_RANK_DECAY |
| content_e2a8dcfda805 |             21877 |           29   | HIGH_EXPOSURE_RANK_DECAY |
| content_df71843dcd17 |             27334 |           76.4 | HIGH_EXPOSURE_RANK_DECAY |


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### 🚨 System Health and Drifting Guardrails
Machine learning models degrade over time as search algorithms evolve and consumer trends change. We establish strict metrics to monitor model health in production:

*   **Primary Health Metric:** **Precision@50** calculated weekly on new prediction results compared against actual 30-day outcomes.
*   **Retrain Triggers (Action required if):**
    1. **Performance Dip:** Precision@50 on production data drops below **60%** for two consecutive weeks.
    2. **Algorithm Shock:** Google rolls out a major, documented Core Update that restructures SERP layouts.
    3. **Data Drift:** Feature distributions (like average baseline CTR or positions across active clients) deviate by more than **20%** from the training distribution baseline.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simulation of a monitoring dashboard trigger checking model drift
current_precision_p50 = 0.58  # Mocking a production performance decline
alert_threshold = 0.60

print("--- System Health Check ---")
print(f"Current Evaluated Precision@50: {current_precision_p50 * 100:.1f}%")
print(f"Critical System Threshold: {alert_threshold * 100:.1f}%")

if current_precision_p50 < alert_threshold:
    print("🚨 SYSTEM ALERT: Model performance has degraded below the critical threshold! Triggering automated pipeline retraining sequence.")
else:
    print("✅ System Healthy: Performance remains within acceptable bounds.")

--- System Health Check ---
Current Evaluated Precision@50: 58.0%
Critical System Threshold: 60.0%
🚨 SYSTEM ALERT: Model performance has degraded below the critical threshold! Triggering automated pipeline retraining sequence.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

To support our final research and capstone paper, we compile and write our highly-vetted, ML-prioritized content refresh queue directly to our local structured storage workspace at `work/outputs/ml_refresh_queue.csv`.

This output contains cleaned metadata, machine learning prediction scores, priority ranks, and actionable reason codes. Our downstream PDF generation engines and research graphs will build directly on top of this exported dataset.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create directory paths
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Format clean export columns for the final queue
export_columns = [
    'priority_rank',
    'content_id',
    'client_id',
    'ml_decay_prob',
    'reason_code',
    'days_since_last_update',
    'impressions_90d',
    'avg_position',
    'ctr',
    'word_count'
]

final_export_df = ranked_queue[export_columns].copy()

# Save final CSV output
output_file = 'work/outputs/ml_refresh_queue.csv'
final_export_df.to_csv(output_file, index=False)

print("--- EXPORT VERIFICATION ---")
print(f"✅ Final ML Priority Queue successfully saved to: {output_file}")
print(f"Export Dimensions: {final_export_df.shape[0]} rows x {final_export_df.shape[1]} columns")
print("\nFirst 3 rows of exported dataset:")
print(final_export_df.head(3).to_markdown(index=False))

--- EXPORT VERIFICATION ---
✅ Final ML Priority Queue successfully saved to: work/outputs/ml_refresh_queue.csv
Export Dimensions: 6163 rows x 10 columns

First 3 rows of exported dataset:
|   priority_rank | content_id           | client_id         |   ml_decay_prob | reason_code         |   days_since_last_update |   impressions_90d |   avg_position |   ctr |   word_count |
|----------------:|:---------------------|:------------------|----------------:|:--------------------|-------------------------:|------------------:|---------------:|------:|-------------:|
|               1 | content_2a228ce7aa1b | client_8527a891e2 |        0.821582 | STANDARD_DECAY_RISK |                       92 |               155 |            7.5 |  0    |         1368 |
|               2 | content_f55fd2d8ed04 | client_4e07408562 |        0.816799 | STANDARD_DECAY_RISK |                      104 |              2237 |            1.3 |  0.09 |         1604 |
|               3 | content_884c401ce126 | client_85

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.